# Data Preparation: Feature Engineering

This notebook mirrors the **Data Preparation** guide at [learnmlops.ops4life.com/guides/data-preparation/](https://learnmlops.ops4life.com/guides/data-preparation/).

Transform the validated employee attrition dataset into a model-ready feature matrix: encode categoricals, aggregate satisfaction signals, bin continuous variables, and engineer risk-indicator features.

**What we'll cover:**
1. Load validated data
2. Define the feature engineering function
3. Apply feature engineering
4. Inspect the result and write `featured.csv`

In [ ]:
import pandas as pd
import numpy as np

## Step 1: Load validated data

In [ ]:
df = pd.read_csv('/workspace/pipeline-data/validated.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

## Step 2: Define the feature engineering function

In [ ]:
def feature_data(df):
    # Work on a copy so the original DataFrame is not mutated.
    # Without .copy(), Python passes a reference and our modifications would
    # silently change the caller's DataFrame — a common source of subtle bugs.
    df_fe = df.copy()

    # ── ENCODING ────────────────────────────────────────────────────────────
    # Machine learning models require numeric inputs. String columns must be
    # converted to numbers before training.

    # Target: convert 'Yes'/'No' string to binary 1/0.
    # This must happen BEFORE any model can use Attrition as a label.
    df_fe['Attrition'] = df_fe['Attrition'].map({'Yes': 1, 'No': 0}).astype('int')

    # Binary encoding: two categories → 0/1 (order is arbitrary for binary).
    # Int64 (capital I) is pandas' nullable integer type — it supports NaN values
    # unlike numpy's int64. Use it when the input might have missing values.
    df_fe['OverTime'] = df_fe['OverTime'].map({'Yes': 1, 'No': 0}).astype('Int64')
    df_fe['Gender']   = df_fe['Gender'].map({'Male': 1, 'Female': 0}).astype('Int64')

    # Ordinal encoding: categories with a natural order get increasing integers.
    # BusinessTravel order (0 < 1 < 2) reflects increasing travel burden.
    # MaritalStatus order (0=Single, 1=Married, 2=Divorced) is arbitrary — no
    # natural ordering exists, but ordinal encoding keeps column count low.
    # Alternative: use pd.get_dummies() for one-hot encoding if ordinality is wrong.
    df_fe['BusinessTravel'] = df_fe['BusinessTravel'].map(
        {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
    ).astype('Int64')
    df_fe['MaritalStatus'] = df_fe['MaritalStatus'].map(
        {'Single': 0, 'Married': 1, 'Divorced': 2}
    ).astype('Int64')

    # ── AGGREGATE SATISFACTION SIGNALS ──────────────────────────────────────
    # Four satisfaction scores (all on a 1–4 scale) are averaged into a single
    # OverallSatisfaction feature. This reduces multicollinearity (the four scores
    # are correlated) and keeps the feature count manageable.
    # .round() converts 2.75 → 3 (nearest integer); .astype('Int64') removes decimals.
    df_fe['OverallSatisfaction'] = (
        (df_fe['JobSatisfaction'] + df_fe['EnvironmentSatisfaction'] +
         df_fe['RelationshipSatisfaction'] + df_fe['WorkLifeBalance']) / 4
    ).round().astype('Int64')

    # Drop the original four columns — they are now represented by OverallSatisfaction.
    # Keeping them alongside the aggregate would give the model redundant signals
    # and inflate feature importance scores.
    df_fe = df_fe.drop(columns=['JobSatisfaction', 'EnvironmentSatisfaction',
                                 'RelationshipSatisfaction', 'WorkLifeBalance'])

    # ── BIN CONTINUOUS VARIABLES ─────────────────────────────────────────────
    # pd.cut() converts a continuous range into ordered categorical bands.
    # Using integer labels (0, 1, 2, ...) instead of strings keeps dtypes numeric.
    # include_lowest=True ensures the minimum value falls into the first bin
    # (by default the leftmost boundary is exclusive).

    # Annual income bands (MonthlyIncome × 12):
    # 0 = <$25k,  1 = $25k–50k,  2 = $50k–75k,  3 = $75k–100k,  4 = >$100k
    df_fe['AnnualIncome'] = pd.cut(
        df_fe['MonthlyIncome'] * 12,
        bins=[0, 25000, 50000, 75000, 100000, float('inf')],
        labels=[0, 1, 2, 3, 4],
        include_lowest=True,
    ).astype('Int64')
    df_fe = df_fe.drop(columns=['MonthlyIncome'])   # Replaced by AnnualIncome

    # Age bands: 1=22–30 (early career), 2=31–40, 3=41–50, 4=51–65 (senior)
    df_fe['AgeGroup'] = pd.cut(
        df_fe['Age'],
        bins=[17, 30, 40, 50, 65],
        labels=[1, 2, 3, 4],
    ).astype('Int64')
    df_fe = df_fe.drop(columns=['Age'])   # Replaced by AgeGroup

    # ── DERIVED RISK-INDICATOR FEATURES ─────────────────────────────────────
    # These features encode domain knowledge about attrition risk. They combine
    # existing columns into new signals that a linear model can capture directly.

    # PromotionStagnation: ratio of years without promotion to total company tenure.
    # High ratio (e.g., 0.8) means the employee has been at the company a long time
    # without advancing — a known attrition predictor.
    # +1 in denominator prevents ZeroDivisionError for employees with YearsAtCompany=0.
    df_fe['PromotionStagnation'] = (
        df_fe['YearsSinceLastPromotion'] / (df_fe['YearsAtCompany'] + 1)
    ).round(3)

    # CareerVelocity: how quickly the employee has advanced relative to total experience.
    # High ratio (e.g., 0.9) means fast promotion = likely retained.
    # Low ratio (e.g., 0.1) means slow career progression = higher attrition risk.
    df_fe['CareerVelocity'] = (
        df_fe['JobLevel'] / (df_fe['TotalWorkingYears'] + 1)
    ).round(3)

    # LongCommute: binary flag for employees living more than 20 units from the office.
    # Long commute is a documented attrition factor (burnout, time cost).
    df_fe['LongCommute'] = (df_fe['DistanceFromHome'] > 20).astype('Int64')

    # StableManager: binary flag for employees who've had the same manager ≥3 years.
    # Manager stability is associated with lower attrition (continuity, trust).
    df_fe['StableManager'] = (df_fe['YearsWithCurrManager'] >= 3).astype('Int64')

    # ── DROP UNINFORMATIVE COLUMNS ───────────────────────────────────────────
    # Remove columns that don't improve model signal:
    # - EmployeeNumber, EmployeeCount, StandardHours, Over18: constants or IDs
    # - JobRole, EducationField, Department: high-cardinality strings that would
    #   need one-hot encoding, adding many sparse columns for marginal gain
    # - DistanceFromHome, YearsWithCurrManager: replaced by derived features above
    # [c for c in [...] if c in df_fe.columns] — safely skips columns that don't exist
    # (e.g., if this function is called on a DataFrame that never had EmployeeCount).
    drop_cols = [c for c in [
        'EmployeeNumber', 'EmployeeCount', 'StandardHours', 'Over18',
        'JobRole', 'EducationField', 'Department',
        'DistanceFromHome', 'YearsWithCurrManager',
    ] if c in df_fe.columns]
    df_fe = df_fe.drop(columns=drop_cols)

    return df_fe


The function applies transformations in four logical groups:
1. **Encoding** — convert string categories to numeric (binary and ordinal)
2. **Aggregation** — collapse four satisfaction scores into `OverallSatisfaction`
3. **Binning** — convert `MonthlyIncome` and `Age` into ordinal bands
4. **Derived features** — engineer signals that correlate with attrition (`PromotionStagnation`, `CareerVelocity`, `LongCommute`, `StableManager`)

## Step 3: Apply feature engineering

In [ ]:
df_featured = feature_data(df)
print(f'Output shape: {df_featured.shape}')
print(f'Columns: {list(df_featured.columns)}')

## Step 4: Inspect the result

In [ ]:
print('Dtypes:')
print(df_featured.dtypes)
print()
df_featured.head()

In [ ]:
output_path = '/workspace/pipeline-data/featured.csv'
df_featured.to_csv(output_path, index=False)
print(f'Written: {output_path}')

## Next Steps

- Continue to preprocessing and train/test split: `02-pipeline/data-preparation/preprocess.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/data-preparation/](https://learnmlops.ops4life.com/guides/data-preparation/)